# Wikidata Person Item Backfill from ORCID

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MattArtzAnthro/wikidata-tools/blob/main/notebooks/Wikidata_ORCID_Person_Backfill.ipynb)

Created by [Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

---

## What This Notebook Does

This notebook takes structured ORCID data and uses it to enrich existing Wikidata person items. It performs a comprehensive comparison between ORCID records and Wikidata items to identify missing data that can be backfilled, then generates QuickStatements commands to add the missing information.

The workflow starts by looking up each ORCID in Wikidata to find the corresponding person item (Q-ID). It then queries all existing properties on that item and compares them against the ORCID data to identify gaps. This enables systematic enrichment of researcher profiles with verified data from ORCID.

## Key Features

- **ORCID Lookup**: Finds Wikidata person items by ORCID identifier (P496)
- **Comprehensive Property Comparison**: Checks external IDs, education, employment, and biographical data
- **Gap Analysis**: Identifies which ORCID data is missing from Wikidata
- **Organization Resolution**: Maps ROR/GRID/Ringgold IDs to Wikidata organization Q-IDs
- **QuickStatements Generation**: Creates batch commands with proper qualifiers and references
- **ORCID as Source**: All additions cite ORCID as the stated source

## Supported Property Mappings

| ORCID Field | Wikidata Property | Notes |
|-------------|-------------------|-------|
| ORCID iD | P496 | Used for lookup |
| Scopus Author ID | P1153 | External identifier |
| ResearcherID | P1053 | External identifier |
| ISNI | P213 | External identifier |
| Loop profile | P2798 | External identifier |
| GitHub username | P2037 | External identifier |
| Google Scholar ID | P1960 | Extracted from URLs |
| ResearchGate ID | P2038 | Extracted from URLs |
| Twitter/X username | P2002 | Extracted from URLs |
| LinkedIn ID | P6634 | Extracted from URLs |
| Academia.edu URL | P5715 | Extracted from URLs |
| Official website | P856 | From URLs |
| Education (with ROR) | P69 | educated at |
| Employment (with ROR) | P108 | employer |
| Country | P27 | country of citizenship |

## Workflow

1. Upload ORCID data (JSON from ORCID_Author_Data_Query notebook)
2. Look up person Q-IDs in Wikidata by ORCID
3. Query existing properties for each person item
4. Compare ORCID data against Wikidata to find gaps
5. Resolve organization identifiers to Wikidata Q-IDs
6. Generate QuickStatements to backfill missing data
7. Export batch file for upload

## Citation

> Artz, M. (2026). Wikidata Tools. GitHub. https://github.com/MattArtzAnthro/wikidata-tools

*A citable DOI will be available via Zenodo.*

## License

[CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/)

## Setup and Installation

*Install required Python packages and import necessary libraries.*

In [ ]:
!pip install requests pandas ipywidgets -q

import requests
import pandas as pd
import json
import re
import time
from datetime import datetime
from collections import defaultdict
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
from io import BytesIO

print("Setup complete.")

## Configuration

*Define Wikidata endpoints, property mappings, and styling.*

In [ ]:
# Wikidata SPARQL endpoint (person items are on main endpoint)
WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"

# User agent for API requests
USER_AGENT = "WikidataORCIDBackfill/1.0 (matt@mattartz.me; Wikidata bot)"

# ORCID Q-ID for references
ORCID_QID = "Q51044"  # ORCID organization/database

# Property IDs - External Identifiers
P496 = "P496"    # ORCID iD
P1153 = "P1153"  # Scopus author ID
P1053 = "P1053"  # ResearcherID
P213 = "P213"    # ISNI
P2798 = "P2798"  # Loop ID
P2037 = "P2037"  # GitHub username
P1960 = "P1960"  # Google Scholar author ID
P2038 = "P2038"  # ResearchGate profile ID
P2002 = "P2002"  # X (Twitter) username
P6634 = "P6634"  # LinkedIn personal profile ID
P5715 = "P5715"  # Academia.edu profile URL

# Property IDs - Person data
P69 = "P69"      # educated at
P108 = "P108"    # employer
P27 = "P27"      # country of citizenship
P106 = "P106"    # occupation
P101 = "P101"    # field of work
P856 = "P856"    # official website

# Qualifier properties
P580 = "P580"    # start time
P582 = "P582"    # end time
P512 = "P512"    # academic degree
P812 = "P812"    # academic major

# Reference properties
S248 = "S248"    # stated in
S854 = "S854"    # reference URL

# Organization ID properties (for lookup)
P6782 = "P6782"  # ROR ID
P2427 = "P2427"  # GRID ID
P3500 = "P3500"  # Ringgold ID

# Mapping of ORCID external ID types to Wikidata properties
EXTERNAL_ID_MAPPING = {
    'Scopus Author ID': P1153,
    'ResearcherID': P1053,
    'ISNI': P213,
    'Loop profile': P2798,
    'GitHub': P2037,
}

# Country code to Q-ID mapping (common countries)
COUNTRY_MAPPING = {
    'US': 'Q30',      # United States
    'GB': 'Q145',     # United Kingdom
    'DE': 'Q183',     # Germany
    'FR': 'Q142',     # France
    'CA': 'Q16',      # Canada
    'AU': 'Q408',     # Australia
    'NL': 'Q55',      # Netherlands
    'IT': 'Q38',      # Italy
    'ES': 'Q29',      # Spain
    'CH': 'Q39',      # Switzerland
    'SE': 'Q34',      # Sweden
    'NO': 'Q20',      # Norway
    'DK': 'Q35',      # Denmark
    'FI': 'Q33',      # Finland
    'JP': 'Q17',      # Japan
    'CN': 'Q148',     # China
    'IN': 'Q668',     # India
    'BR': 'Q155',     # Brazil
    'MX': 'Q96',      # Mexico
    'ZA': 'Q258',     # South Africa
}

# Color palette
COLORS = {
    'bg_primary': '#E7ECEF',
    'text_primary': '#274C77',
    'interactive': '#6096BA',
    'bg_secondary': '#A3CEF1',
    'neutral': '#8B8C89',
    'success': '#28a745',
    'warning': '#ffc107'
}

CONTAINER_STYLE = f"""
    background-color: {COLORS['bg_primary']};
    border-left: 5px solid {COLORS['text_primary']};
    border-radius: 10px;
    padding: 15px;
    margin: 10px 0;
"""

print("Configuration loaded.")
print(f"External ID mappings: {len(EXTERNAL_ID_MAPPING)}")
print(f"Country mappings: {len(COUNTRY_MAPPING)}")

## Helper Functions: SPARQL Queries

*Functions to query Wikidata for person items and their properties.*

In [ ]:
def sparql_query(query, timeout=60):
    """Execute a SPARQL query against Wikidata and return results."""
    try:
        response = requests.get(
            WIKIDATA_ENDPOINT,
            params={"query": query, "format": "json"},
            headers={"User-Agent": USER_AGENT},
            timeout=timeout
        )
        response.raise_for_status()
        return response.json().get("results", {}).get("bindings", [])
    except requests.exceptions.Timeout:
        print(f"   Query timeout")
        return []
    except requests.exceptions.RequestException as e:
        print(f"   Request error: {e}")
        return []
    except Exception as e:
        print(f"   Unexpected error: {e}")
        return []


def clean_orcid(orcid_input):
    """
    Extract clean ORCID from various formats.
    Returns 16-digit ORCID or None.
    """
    if not orcid_input or not isinstance(orcid_input, str):
        return None
    
    orcid_str = str(orcid_input).strip()
    
    # Extract from URL format
    if 'orcid.org/' in orcid_str:
        orcid_str = orcid_str.split('orcid.org/')[-1].strip(']').strip()
    
    # Validate format (0000-0000-0000-000X)
    orcid_pattern = r'^\d{4}-\d{4}-\d{4}-\d{3}[\dX]$'
    if re.match(orcid_pattern, orcid_str):
        return orcid_str
    
    return None


def lookup_person_by_orcid(orcid):
    """
    Look up a person in Wikidata by ORCID.
    Returns QID if found, None otherwise.
    """
    orcid_clean = clean_orcid(orcid)
    if not orcid_clean:
        return None

    query = f"""
    SELECT ?person ?personLabel WHERE {{
      ?person wdt:P496 "{orcid_clean}" .
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    LIMIT 1
    """

    results = sparql_query(query)
    
    if results:
        qid = results[0].get('person', {}).get('value', '').split('/')[-1]
        label = results[0].get('personLabel', {}).get('value', '')
        return {'qid': qid, 'label': label}
    return None


def get_person_external_ids(qid):
    """
    Get all external identifiers for a person.
    Returns dict mapping property ID to list of values.
    """
    # Include all supported external ID properties
    query = f"""
    SELECT ?prop ?propLabel ?value WHERE {{
      VALUES ?prop {{ 
        wd:P496 wd:P1153 wd:P1053 wd:P213 wd:P2798 wd:P2037
        wd:P1960 wd:P2038 wd:P2002 wd:P6634 wd:P5715 wd:P856
      }}
      OPTIONAL {{ wd:{qid} ?propDirect ?value . ?prop wikibase:directClaim ?propDirect . }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """
    
    results = sparql_query(query)
    
    ids = defaultdict(list)
    for r in results:
        prop_uri = r.get('prop', {}).get('value', '')
        prop_id = prop_uri.split('/')[-1] if prop_uri else None
        value = r.get('value', {}).get('value', '')
        if prop_id and value:
            ids[prop_id].append(value)
    
    return dict(ids)


def get_person_affiliations(qid):
    """
    Get education and employment for a person.
    Returns dict with 'educated_at' and 'employer' lists of Q-IDs.
    """
    query = f"""
    SELECT ?educatedAt ?educatedAtLabel ?employer ?employerLabel WHERE {{
      OPTIONAL {{ wd:{qid} wdt:P69 ?educatedAt }}
      OPTIONAL {{ wd:{qid} wdt:P108 ?employer }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """
    
    results = sparql_query(query)
    
    affiliations = {
        'educated_at': set(),
        'employer': set()
    }
    
    for r in results:
        edu_uri = r.get('educatedAt', {}).get('value', '')
        if edu_uri:
            affiliations['educated_at'].add(edu_uri.split('/')[-1])
        
        emp_uri = r.get('employer', {}).get('value', '')
        if emp_uri:
            affiliations['employer'].add(emp_uri.split('/')[-1])
    
    return {
        'educated_at': list(affiliations['educated_at']),
        'employer': list(affiliations['employer'])
    }


def get_person_countries(qid):
    """
    Get country of citizenship for a person.
    Returns list of country Q-IDs.
    """
    query = f"""
    SELECT ?country WHERE {{
      wd:{qid} wdt:P27 ?country .
    }}
    """
    
    results = sparql_query(query)
    
    countries = []
    for r in results:
        uri = r.get('country', {}).get('value', '')
        if uri:
            countries.append(uri.split('/')[-1])
    
    return countries


print("SPARQL query functions loaded.")

## Helper Functions: Organization Resolution

*Functions to resolve ROR/GRID/Ringgold IDs to Wikidata Q-IDs.*

In [ ]:
# Cache for organization lookups
org_cache = {}

def extract_ror_id(ror_url):
    """
    Extract ROR ID from URL format.
    https://ror.org/00b30xv10 -> 00b30xv10
    """
    if not ror_url:
        return None
    if 'ror.org/' in str(ror_url):
        return str(ror_url).split('ror.org/')[-1].strip()
    return str(ror_url).strip()


def lookup_org_by_ror(ror_id):
    """
    Look up an organization in Wikidata by ROR ID.
    """
    ror_clean = extract_ror_id(ror_id)
    if not ror_clean:
        return None
    
    cache_key = f"ror:{ror_clean}"
    if cache_key in org_cache:
        return org_cache[cache_key]
    
    query = f"""
    SELECT ?org ?orgLabel WHERE {{
      ?org wdt:P6782 "{ror_clean}" .
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    LIMIT 1
    """
    
    results = sparql_query(query)
    
    if results:
        qid = results[0].get('org', {}).get('value', '').split('/')[-1]
        label = results[0].get('orgLabel', {}).get('value', '')
        result = {'qid': qid, 'label': label}
        org_cache[cache_key] = result
        return result
    
    org_cache[cache_key] = None
    return None


def lookup_org_by_grid(grid_id):
    """
    Look up an organization in Wikidata by GRID ID.
    """
    if not grid_id:
        return None
    
    # Extract GRID ID from various formats
    grid_clean = str(grid_id).strip()
    if 'grid.' in grid_clean:
        # Already in GRID format
        pass
    
    cache_key = f"grid:{grid_clean}"
    if cache_key in org_cache:
        return org_cache[cache_key]
    
    query = f"""
    SELECT ?org ?orgLabel WHERE {{
      ?org wdt:P2427 "{grid_clean}" .
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    LIMIT 1
    """
    
    results = sparql_query(query)
    
    if results:
        qid = results[0].get('org', {}).get('value', '').split('/')[-1]
        label = results[0].get('orgLabel', {}).get('value', '')
        result = {'qid': qid, 'label': label}
        org_cache[cache_key] = result
        return result
    
    org_cache[cache_key] = None
    return None


def lookup_org_by_ringgold(ringgold_id):
    """
    Look up an organization in Wikidata by Ringgold ID.
    """
    if not ringgold_id:
        return None
    
    ringgold_clean = str(ringgold_id).strip()
    
    cache_key = f"ringgold:{ringgold_clean}"
    if cache_key in org_cache:
        return org_cache[cache_key]
    
    query = f"""
    SELECT ?org ?orgLabel WHERE {{
      ?org wdt:P3500 "{ringgold_clean}" .
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    LIMIT 1
    """
    
    results = sparql_query(query)
    
    if results:
        qid = results[0].get('org', {}).get('value', '').split('/')[-1]
        label = results[0].get('orgLabel', {}).get('value', '')
        result = {'qid': qid, 'label': label}
        org_cache[cache_key] = result
        return result
    
    org_cache[cache_key] = None
    return None


def resolve_organization(disambiguated_id, disambiguated_source):
    """
    Resolve an organization identifier to a Wikidata Q-ID.
    Handles ROR, GRID, Ringgold, and FUNDREF sources.
    """
    if not disambiguated_id or not disambiguated_source:
        return None
    
    source = str(disambiguated_source).upper()
    
    if source == 'ROR':
        return lookup_org_by_ror(disambiguated_id)
    elif source == 'GRID':
        return lookup_org_by_grid(disambiguated_id)
    elif source == 'RINGGOLD':
        return lookup_org_by_ringgold(disambiguated_id)
    elif source == 'FUNDREF':
        # FUNDREF uses DOI-like identifiers, try extracting
        return None  # TODO: implement FUNDREF lookup
    
    return None


print("Organization resolution functions loaded.")

## Helper Functions: URL Parsing

*Functions to extract external identifiers from ORCID URLs field.*

In [ ]:
def extract_ids_from_urls(urls):
    """
    Extract external identifiers from ORCID URLs field.
    
    Args:
        urls: List of URL dicts with 'name' and 'url' keys, or list of URL strings
    
    Returns:
        Dict mapping identifier type to value
    """
    extracted = {}
    
    if not urls:
        return extracted
    
    # Normalize to list of URL strings
    url_list = []
    for item in urls:
        if isinstance(item, dict):
            url_list.append(item.get('url', ''))
        elif isinstance(item, str):
            url_list.append(item)
    
    for url in url_list:
        if not url:
            continue
        url_lower = url.lower()
        
        # Google Scholar
        # Format: https://scholar.google.com/citations?user=XXXX
        if 'scholar.google.com' in url_lower and 'user=' in url_lower:
            match = re.search(r'user=([^&]+)', url)
            if match:
                extracted['Google Scholar author ID'] = match.group(1)
        
        # ResearchGate
        # Format: https://www.researchgate.net/profile/Name_Name
        elif 'researchgate.net/profile/' in url_lower:
            match = re.search(r'researchgate\.net/profile/([^/?]+)', url, re.IGNORECASE)
            if match:
                extracted['ResearchGate profile ID'] = match.group(1)
        
        # Twitter/X
        # Format: https://twitter.com/username or https://x.com/username
        elif 'twitter.com/' in url_lower or 'x.com/' in url_lower:
            match = re.search(r'(?:twitter|x)\.com/([^/?]+)', url, re.IGNORECASE)
            if match:
                username = match.group(1)
                if username.lower() not in ['home', 'search', 'explore', 'settings', 'i']:
                    extracted['Twitter username'] = username
        
        # LinkedIn
        # Format: https://www.linkedin.com/in/profile-name/
        elif 'linkedin.com/in/' in url_lower:
            match = re.search(r'linkedin\.com/in/([^/?]+)', url, re.IGNORECASE)
            if match:
                extracted['LinkedIn personal profile ID'] = match.group(1).rstrip('/')
        
        # Academia.edu
        # Format: https://xxx.academia.edu/Name
        elif 'academia.edu' in url_lower:
            # Store the full URL as this property takes URLs
            extracted['Academia.edu profile URL'] = url
        
        # Official website (personal sites - often has name in URL)
        # Skip common social media and academic profile sites
        elif not any(site in url_lower for site in [
            'scholar.google', 'researchgate', 'twitter', 'x.com', 'linkedin',
            'academia.edu', 'orcid.org', 'github.com', 'impactstory.org'
        ]):
            # Could be a personal website
            if 'official website' not in extracted:
                extracted['official website'] = url
    
    return extracted


# Extended external ID mapping including URL-extracted IDs
EXTENDED_ID_MAPPING = {
    # From ORCID external_ids field
    'Scopus Author ID': P1153,
    'ResearcherID': P1053,
    'ISNI': P213,
    'Loop profile': P2798,
    'GitHub': P2037,
    # From URL parsing
    'Google Scholar author ID': P1960,
    'ResearchGate profile ID': P2038,
    'Twitter username': P2002,
    'LinkedIn personal profile ID': P6634,
    'Academia.edu profile URL': P5715,
    'official website': P856,
}


print("URL parsing functions loaded.")
print(f"Extended ID mappings: {len(EXTENDED_ID_MAPPING)}")

## Test Endpoint Connection

*Verify Wikidata endpoint is accessible and test lookups.*

In [ ]:
print("Testing Wikidata connection...")
print()

# Test ORCID lookup
print("1. ORCID lookup test:")
test_orcid = "0000-0002-3822-1429"
result = lookup_person_by_orcid(test_orcid)
if result:
    print(f"   ✓ Found: {result['label']} ({result['qid']})")
    test_qid = result['qid']
else:
    print(f"   ✗ ORCID {test_orcid} not found in Wikidata")
    test_qid = None

print()

# Test external ID retrieval
if test_qid:
    print("2. External identifiers test:")
    ext_ids = get_person_external_ids(test_qid)
    if ext_ids:
        print(f"   ✓ Found {len(ext_ids)} identifier types")
        for prop, values in ext_ids.items():
            print(f"      {prop}: {values}")
    else:
        print("   No external identifiers found")
    
    print()
    
    print("3. Affiliations test:")
    affiliations = get_person_affiliations(test_qid)
    print(f"   Education: {len(affiliations['educated_at'])} institution(s)")
    print(f"   Employment: {len(affiliations['employer'])} employer(s)")

print()

# Test ROR lookup
print("4. Organization (ROR) lookup test:")
test_ror = "https://ror.org/00b30xv10"  # University of Pennsylvania
org_result = lookup_org_by_ror(test_ror)
if org_result:
    print(f"   ✓ Found: {org_result['label']} ({org_result['qid']})")
else:
    print(f"   ✗ ROR {test_ror} not found")

print()
print("Connection tests complete.")

## Upload ORCID Data

*Upload ORCID data exported from the ORCID_Author_Data_Query notebook.*

In [ ]:
orcid_records = []  # List of parsed ORCID records

# File upload widget
file_upload = widgets.FileUpload(
    accept='.json,.csv',
    multiple=True,
    description='Upload Files'
)

upload_output = widgets.Output()

def parse_orcid_json(content):
    """Parse ORCID full JSON export."""
    try:
        data = json.loads(content)
        if isinstance(data, list):
            return data
        elif isinstance(data, dict):
            return [data]
    except:
        return []
    return []


def parse_orcid_csv(content):
    """Parse ORCID data CSV and convert to records format."""
    try:
        df = pd.read_csv(BytesIO(content))
        records = []
        
        for _, row in df.iterrows():
            record = {
                'orcid': row.get('orcid', ''),
                'full_name': row.get('full_name', ''),
                'external_ids': {},
                'countries': [],
                'employments': [],
                'educations': []
            }
            
            # Parse external IDs from columns
            if pd.notna(row.get('scopus_id')):
                record['external_ids']['Scopus Author ID'] = str(row['scopus_id'])
            if pd.notna(row.get('researcher_id')):
                record['external_ids']['ResearcherID'] = str(row['researcher_id'])
            if pd.notna(row.get('isni')):
                record['external_ids']['ISNI'] = str(row['isni'])
            if pd.notna(row.get('loop_profile')):
                record['external_ids']['Loop profile'] = str(row['loop_profile'])
            
            # Parse countries
            if pd.notna(row.get('countries')):
                record['countries'] = [c.strip() for c in str(row['countries']).split(';')]
            
            records.append(record)
        
        return records
    except Exception as e:
        print(f"Error parsing CSV: {e}")
        return []


def parse_affiliations_csv(content):
    """Parse ORCID affiliations CSV and merge with existing records."""
    try:
        df = pd.read_csv(BytesIO(content))
        
        # Group by ORCID
        affiliations_by_orcid = defaultdict(lambda: {'employments': [], 'educations': []})
        
        for _, row in df.iterrows():
            orcid = row.get('orcid', '')
            if not orcid:
                continue
            
            aff_type = row.get('affiliation_type', '')
            
            affiliation = {
                'organization_name': row.get('organization_name', ''),
                'disambiguated_org_id': row.get('disambiguated_org_id', ''),
                'disambiguated_org_source': row.get('disambiguated_org_source', ''),
                'role_title': row.get('role_title', ''),
                'department': row.get('department', ''),
                'start_date': row.get('start_date', ''),
                'end_date': row.get('end_date', ''),
                'is_current': row.get('is_current', '') == 'true'
            }
            
            if aff_type == 'employment':
                affiliations_by_orcid[orcid]['employments'].append(affiliation)
            elif aff_type == 'education':
                affiliations_by_orcid[orcid]['educations'].append(affiliation)
        
        return dict(affiliations_by_orcid)
    except Exception as e:
        print(f"Error parsing affiliations CSV: {e}")
        return {}


def on_upload(change):
    global orcid_records
    orcid_records = []
    affiliations_data = {}
    
    with upload_output:
        clear_output()
        
        if not file_upload.value:
            return
        
        print(f"Processing {len(file_upload.value)} file(s)...")
        print()
        
        for filename, file_info in file_upload.value.items():
            content = file_info['content']
            
            if filename.endswith('.json'):
                print(f"📄 {filename} (JSON)")
                records = parse_orcid_json(content)
                orcid_records.extend(records)
                print(f"   Loaded {len(records)} ORCID record(s)")
            
            elif 'affiliations' in filename.lower() and filename.endswith('.csv'):
                print(f"📄 {filename} (Affiliations CSV)")
                affiliations_data = parse_affiliations_csv(content)
                print(f"   Loaded affiliations for {len(affiliations_data)} ORCID(s)")
            
            elif filename.endswith('.csv'):
                print(f"📄 {filename} (Data CSV)")
                records = parse_orcid_csv(content)
                orcid_records.extend(records)
                print(f"   Loaded {len(records)} ORCID record(s)")
        
        # Merge affiliations into records
        if affiliations_data:
            for record in orcid_records:
                orcid = record.get('orcid', '')
                if orcid in affiliations_data:
                    if not record.get('employments'):
                        record['employments'] = affiliations_data[orcid]['employments']
                    if not record.get('educations'):
                        record['educations'] = affiliations_data[orcid]['educations']
        
        print()
        print(f"Total ORCID records: {len(orcid_records)}")
        
        if orcid_records:
            print()
            print("Sample records:")
            for r in orcid_records[:3]:
                ext_count = len(r.get('external_ids', {}))
                emp_count = len(r.get('employments', []))
                edu_count = len(r.get('educations', []))
                print(f"  {r.get('orcid', 'N/A')}: {r.get('full_name', 'N/A')}")
                print(f"    External IDs: {ext_count}, Employments: {emp_count}, Educations: {edu_count}")

file_upload.observe(on_upload, names='value')

# Display
display(HTML(f"""
<div style="{CONTAINER_STYLE}">
    <h3 style="color: {COLORS['text_primary']}; margin-top: 0;">📁 Upload ORCID Data</h3>
    <p>Upload files from the ORCID_Author_Data_Query notebook:</p>
    <ul>
        <li><strong>orcid_full_*.json</strong> - Complete ORCID records (preferred)</li>
        <li><strong>orcid_data_*.csv</strong> - Basic ORCID data</li>
        <li><strong>orcid_affiliations_*.csv</strong> - Education/employment details</li>
    </ul>
    <p><em>You can upload multiple files at once.</em></p>
</div>
"""))
display(file_upload)
display(upload_output)

## Analyze Wikidata vs ORCID

*For each ORCID, compare Wikidata data against ORCID data to identify gaps.*

In [ ]:
# Store analysis results
analysis_results = {}  # orcid -> analysis data

analyze_button = widgets.Button(
    description='Analyze Gaps',
    button_style='primary',
    icon='search'
)

progress_bar = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description='Progress:',
    bar_style='info'
)

analyze_output = widgets.Output()

def run_analysis(button):
    global analysis_results
    analysis_results = {}
    
    with analyze_output:
        clear_output()
        
        if not orcid_records:
            print("Please upload ORCID data first.")
            return
        
        total = len(orcid_records)
        progress_bar.max = total
        progress_bar.value = 0
        
        print(f"Analyzing {total} ORCID record(s)...")
        print("=" * 60)
        print()
        
        found_in_wikidata = 0
        not_found = 0
        total_gaps = 0
        
        for idx, record in enumerate(orcid_records):
            progress_bar.value = idx + 1
            
            orcid = clean_orcid(record.get('orcid', ''))
            if not orcid:
                continue
            
            full_name = record.get('full_name', orcid)
            
            # Look up person in Wikidata
            person = lookup_person_by_orcid(orcid)
            time.sleep(0.3)
            
            if not person:
                not_found += 1
                print(f"[{idx+1}/{total}] {full_name} - NOT in Wikidata")
                continue
            
            found_in_wikidata += 1
            qid = person['qid']
            
            print(f"[{idx+1}/{total}] {full_name} ({qid}) - Analyzing...")
            
            # Get current Wikidata data
            wd_external_ids = get_person_external_ids(qid)
            wd_affiliations = get_person_affiliations(qid)
            wd_countries = get_person_countries(qid)
            time.sleep(0.5)
            
            # Find gaps
            gaps = {
                'external_ids': [],
                'educations': [],
                'employments': [],
                'countries': []
            }
            
            # Check external IDs from ORCID external_ids field
            orcid_ext_ids = record.get('external_ids', {})
            for id_type, id_value in orcid_ext_ids.items():
                if id_type in EXTENDED_ID_MAPPING:
                    wd_prop = EXTENDED_ID_MAPPING[id_type]
                    if wd_prop not in wd_external_ids or id_value not in wd_external_ids[wd_prop]:
                        gaps['external_ids'].append({
                            'type': id_type,
                            'property': wd_prop,
                            'value': id_value
                        })
            
            # Extract and check IDs from URLs field
            url_ids = extract_ids_from_urls(record.get('urls', []))
            for id_type, id_value in url_ids.items():
                if id_type in EXTENDED_ID_MAPPING:
                    wd_prop = EXTENDED_ID_MAPPING[id_type]
                    # Check if this value already exists in Wikidata
                    existing_values = wd_external_ids.get(wd_prop, [])
                    # For URLs, do a looser match (the URL might be slightly different)
                    value_exists = any(id_value.lower() in ev.lower() or ev.lower() in id_value.lower() 
                                       for ev in existing_values)
                    if not value_exists:
                        gaps['external_ids'].append({
                            'type': id_type,
                            'property': wd_prop,
                            'value': id_value
                        })
            
            # Check education
            for edu in record.get('educations', []):
                org_id = edu.get('disambiguated_org_id')
                org_source = edu.get('disambiguated_org_source')
                
                if org_id and org_source:
                    org_qid_result = resolve_organization(org_id, org_source)
                    if org_qid_result:
                        org_qid = org_qid_result['qid']
                        if org_qid not in wd_affiliations['educated_at']:
                            gaps['educations'].append({
                                'org_qid': org_qid,
                                'org_label': org_qid_result['label'],
                                'org_name': edu.get('organization_name', ''),
                                'role': edu.get('role_title', ''),
                                'department': edu.get('department', ''),
                                'start_date': edu.get('start_date', ''),
                                'end_date': edu.get('end_date', '')
                            })
                    time.sleep(0.2)
            
            # Check employment
            for emp in record.get('employments', []):
                org_id = emp.get('disambiguated_org_id')
                org_source = emp.get('disambiguated_org_source')
                
                if org_id and org_source:
                    org_qid_result = resolve_organization(org_id, org_source)
                    if org_qid_result:
                        org_qid = org_qid_result['qid']
                        if org_qid not in wd_affiliations['employer']:
                            gaps['employments'].append({
                                'org_qid': org_qid,
                                'org_label': org_qid_result['label'],
                                'org_name': emp.get('organization_name', ''),
                                'role': emp.get('role_title', ''),
                                'department': emp.get('department', ''),
                                'start_date': emp.get('start_date', ''),
                                'end_date': emp.get('end_date', ''),
                                'is_current': emp.get('is_current', False)
                            })
                    time.sleep(0.2)
            
            # Check countries
            for country_code in record.get('countries', []):
                if country_code in COUNTRY_MAPPING:
                    country_qid = COUNTRY_MAPPING[country_code]
                    if country_qid not in wd_countries:
                        gaps['countries'].append({
                            'code': country_code,
                            'qid': country_qid
                        })
            
            # Count gaps
            gap_count = sum(len(v) for v in gaps.values())
            total_gaps += gap_count
            
            if gap_count > 0:
                analysis_results[orcid] = {
                    'qid': qid,
                    'name': full_name,
                    'gaps': gaps
                }
                print(f"   Found {gap_count} gap(s): "
                      f"{len(gaps['external_ids'])} IDs, "
                      f"{len(gaps['educations'])} edu, "
                      f"{len(gaps['employments'])} emp, "
                      f"{len(gaps['countries'])} country")
            else:
                print(f"   No gaps found - Wikidata is complete!")
        
        print()
        print("=" * 60)
        print("ANALYSIS SUMMARY")
        print("=" * 60)
        print(f"Total ORCIDs processed: {total}")
        print(f"Found in Wikidata: {found_in_wikidata}")
        print(f"Not in Wikidata: {not_found}")
        print(f"With gaps to fill: {len(analysis_results)}")
        print(f"Total gaps identified: {total_gaps}")

analyze_button.on_click(run_analysis)

# Display
display(HTML(f"""
<div style="{CONTAINER_STYLE}">
    <h3 style="color: {COLORS['text_primary']}; margin-top: 0;">🔍 Analyze Gaps</h3>
    <p>Compare ORCID data against Wikidata to identify missing information.</p>
    <p><em>This will look up each ORCID in Wikidata and compare properties.</em></p>
</div>
"""))
display(widgets.VBox([
    analyze_button,
    progress_bar,
    analyze_output
]))

## Review Gaps

*Review the identified gaps before generating QuickStatements.*

In [ ]:
review_output = widgets.Output()

def show_review():
    with review_output:
        clear_output()
        
        if not analysis_results:
            print("No gaps to review. Run analysis first.")
            return
        
        total_gaps = sum(
            sum(len(v) for v in data['gaps'].values())
            for data in analysis_results.values()
        )
        
        print(f"GAPS TO BACKFILL: {total_gaps} across {len(analysis_results)} person(s)")
        print("=" * 70)
        print()
        
        for orcid, data in analysis_results.items():
            print(f"👤 {data['name']} ({data['qid']})")
            print(f"   ORCID: {orcid}")
            print()
            
            gaps = data['gaps']
            
            # External IDs
            if gaps['external_ids']:
                print("   📎 External Identifiers to add:")
                for ext_id in gaps['external_ids']:
                    print(f"      + {ext_id['type']} ({ext_id['property']}): {ext_id['value']}")
                print()
            
            # Education
            if gaps['educations']:
                print("   🎓 Education to add (P69):")
                for edu in gaps['educations']:
                    date_str = ""
                    if edu.get('end_date'):
                        date_str = f" ({edu['end_date']})"
                    print(f"      + {edu['org_label']} ({edu['org_qid']}){date_str}")
                    if edu.get('role'):
                        print(f"        Degree: {edu['role']}")
                print()
            
            # Employment
            if gaps['employments']:
                print("   💼 Employment to add (P108):")
                for emp in gaps['employments']:
                    current = " [current]" if emp.get('is_current') else ""
                    print(f"      + {emp['org_label']} ({emp['org_qid']}){current}")
                    if emp.get('role'):
                        print(f"        Position: {emp['role']}")
                print()
            
            # Countries
            if gaps['countries']:
                print("   🌍 Country to add (P27):")
                for country in gaps['countries']:
                    print(f"      + {country['code']} ({country['qid']})")
                print()
            
            print("-" * 70)
            print()

review_button = widgets.Button(
    description='Review Gaps',
    button_style='info',
    icon='eye'
)
review_button.on_click(lambda b: show_review())

display(HTML(f"""
<div style="{CONTAINER_STYLE}">
    <h3 style="color: {COLORS['text_primary']}; margin-top: 0;">👀 Review Gaps</h3>
    <p>Review all identified gaps before generating QuickStatements.</p>
</div>
"""))
display(review_button)
display(review_output)

## Generate QuickStatements

*Generate QuickStatements to backfill missing data from ORCID.*

In [ ]:
def escape_qs_string(s):
    """Escape a string for QuickStatements V1 format."""
    if not s:
        return s
    return str(s).replace('"', '""')


def format_date_for_qs(date_str):
    """
    Format a date string for QuickStatements.
    Handles: YYYY, YYYY-MM, YYYY-MM-DD
    Returns +YYYY-MM-DDT00:00:00Z/precision format
    """
    if not date_str:
        return None
    
    date_str = str(date_str).strip()
    
    # Try different formats
    if re.match(r'^\d{4}$', date_str):
        # Year only (precision 9)
        return f"+{date_str}-00-00T00:00:00Z/9"
    elif re.match(r'^\d{4}-\d{2}$', date_str):
        # Year-month (precision 10)
        return f"+{date_str}-00T00:00:00Z/10"
    elif re.match(r'^\d{4}-\d{2}-\d{2}$', date_str):
        # Full date (precision 11)
        return f"+{date_str}T00:00:00Z/11"
    
    return None


def generate_quickstatements(include_external_ids=True, include_education=True,
                            include_employment=True, include_countries=False):
    """
    Generate QuickStatements V1 format to backfill Wikidata from ORCID.
    All statements include ORCID as the stated source.
    """
    qs_lines = []
    
    for orcid, data in analysis_results.items():
        person_qid = data['qid']
        gaps = data['gaps']
        
        # ORCID reference for all statements
        orcid_ref = f"|{S248}|{ORCID_QID}|{S854}|\"https://orcid.org/{orcid}\""
        
        # External identifiers
        if include_external_ids:
            for ext_id in gaps['external_ids']:
                value_escaped = escape_qs_string(ext_id['value'])
                qs_lines.append(f"{person_qid}|{ext_id['property']}|\"{value_escaped}\"{orcid_ref}")
        
        # Education (P69)
        if include_education:
            for edu in gaps['educations']:
                parts = [person_qid, P69, edu['org_qid']]
                
                # Add end time if available (graduation date)
                if edu.get('end_date'):
                    date_formatted = format_date_for_qs(edu['end_date'])
                    if date_formatted:
                        parts.extend([P582, date_formatted])
                
                # Add start time if available
                if edu.get('start_date'):
                    date_formatted = format_date_for_qs(edu['start_date'])
                    if date_formatted:
                        parts.extend([P580, date_formatted])
                
                # Add reference
                parts.append(f"{S248}|{ORCID_QID}|{S854}|\"https://orcid.org/{orcid}\"")
                
                qs_lines.append('|'.join(parts))
        
        # Employment (P108)
        if include_employment:
            for emp in gaps['employments']:
                parts = [person_qid, P108, emp['org_qid']]
                
                # Add start time
                if emp.get('start_date'):
                    date_formatted = format_date_for_qs(emp['start_date'])
                    if date_formatted:
                        parts.extend([P580, date_formatted])
                
                # Add end time (if not current)
                if emp.get('end_date') and not emp.get('is_current'):
                    date_formatted = format_date_for_qs(emp['end_date'])
                    if date_formatted:
                        parts.extend([P582, date_formatted])
                
                # Add reference
                parts.append(f"{S248}|{ORCID_QID}|{S854}|\"https://orcid.org/{orcid}\"")
                
                qs_lines.append('|'.join(parts))
        
        # Countries (P27) - disabled by default as it may not be appropriate
        if include_countries:
            for country in gaps['countries']:
                qs_lines.append(f"{person_qid}|{P27}|{country['qid']}{orcid_ref}")
    
    return '\n'.join(qs_lines)


# Option checkboxes
include_ext_ids_cb = widgets.Checkbox(
    value=True,
    description='Include external identifiers (Scopus, ResearcherID, ISNI, etc.)',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

include_education_cb = widgets.Checkbox(
    value=True,
    description='Include education (P69 - educated at)',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

include_employment_cb = widgets.Checkbox(
    value=True,
    description='Include employment (P108 - employer)',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

include_countries_cb = widgets.Checkbox(
    value=False,
    description='Include country of citizenship (P27) - use with caution',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

generate_button = widgets.Button(
    description='Generate QuickStatements',
    button_style='success',
    icon='download'
)

generate_output = widgets.Output()

def run_generate(button):
    with generate_output:
        clear_output()
        
        if not analysis_results:
            print("No gaps to fill. Run analysis first.")
            return
        
        print("Generating QuickStatements...")
        print()
        
        qs_text = generate_quickstatements(
            include_external_ids=include_ext_ids_cb.value,
            include_education=include_education_cb.value,
            include_employment=include_employment_cb.value,
            include_countries=include_countries_cb.value
        )
        
        if not qs_text:
            print("No QuickStatements generated.")
            return
        
        # Save to file
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"orcid_backfill_{timestamp}.txt"
        
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(qs_text)
        
        line_count = len(qs_text.strip().split('\n'))
        
        print(f"Generated {line_count} QuickStatements commands")
        print(f"For {len(analysis_results)} person item(s)")
        print(f"Saved to: {filename}")
        print()
        
        # Preview
        print("--- PREVIEW (first 5 commands) ---")
        preview_lines = qs_text.strip().split('\n')[:5]
        for line in preview_lines:
            if len(line) > 100:
                print(line[:100] + "...")
            else:
                print(line)
        print()
        
        print("--- PROPERTIES BEING ADDED ---")
        if include_ext_ids_cb.value:
            print("  External IDs: P1153 (Scopus), P1053 (ResearcherID), P213 (ISNI), etc.")
        if include_education_cb.value:
            print("  Education: P69 (educated at) with date qualifiers")
        if include_employment_cb.value:
            print("  Employment: P108 (employer) with date qualifiers")
        if include_countries_cb.value:
            print("  Country: P27 (country of citizenship)")
        print()
        print("All statements include ORCID as stated source (S248/S854)")
        print()
        
        print("--- UPLOAD INSTRUCTIONS ---")
        print("1. Go to: https://quickstatements.toolforge.org/")
        print("2. Log in with your Wikidata account")
        print("3. Click 'New batch'")
        print("4. Paste the file contents")
        print("5. Click 'Import V1 commands'")
        print("6. Review carefully and click 'Run'")
        
        # Download in Colab
        try:
            from google.colab import files
            files.download(filename)
        except:
            pass

generate_button.on_click(run_generate)

# Display
display(HTML(f"""
<div style="{CONTAINER_STYLE}">
    <h3 style="color: {COLORS['text_primary']}; margin-top: 0;">📤 Generate QuickStatements</h3>
    <p>Generate QuickStatements to backfill missing data from ORCID.</p>
    <p><strong>Select which data to include:</strong></p>
</div>
"""))
display(widgets.VBox([
    include_ext_ids_cb,
    include_education_cb,
    include_employment_cb,
    include_countries_cb,
    generate_button,
    generate_output
]))

## Export Summary

*Export a CSV summary of all identified gaps.*

In [ ]:
export_button = widgets.Button(
    description='Export Gap Summary',
    button_style='info',
    icon='table'
)

export_output = widgets.Output()

def run_export(button):
    with export_output:
        clear_output()
        
        if not analysis_results:
            print("No data to export. Run analysis first.")
            return
        
        rows = []
        for orcid, data in analysis_results.items():
            gaps = data['gaps']
            
            # External IDs
            for ext_id in gaps['external_ids']:
                rows.append({
                    'ORCID': orcid,
                    'Person_QID': data['qid'],
                    'Person_Name': data['name'],
                    'Gap_Type': 'External ID',
                    'Property': ext_id['property'],
                    'Value': ext_id['value'],
                    'Details': ext_id['type']
                })
            
            # Education
            for edu in gaps['educations']:
                rows.append({
                    'ORCID': orcid,
                    'Person_QID': data['qid'],
                    'Person_Name': data['name'],
                    'Gap_Type': 'Education',
                    'Property': 'P69',
                    'Value': edu['org_qid'],
                    'Details': f"{edu['org_label']} - {edu.get('role', '')}"
                })
            
            # Employment
            for emp in gaps['employments']:
                rows.append({
                    'ORCID': orcid,
                    'Person_QID': data['qid'],
                    'Person_Name': data['name'],
                    'Gap_Type': 'Employment',
                    'Property': 'P108',
                    'Value': emp['org_qid'],
                    'Details': f"{emp['org_label']} - {emp.get('role', '')}"
                })
            
            # Countries
            for country in gaps['countries']:
                rows.append({
                    'ORCID': orcid,
                    'Person_QID': data['qid'],
                    'Person_Name': data['name'],
                    'Gap_Type': 'Country',
                    'Property': 'P27',
                    'Value': country['qid'],
                    'Details': country['code']
                })
        
        df = pd.DataFrame(rows)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"orcid_gaps_summary_{timestamp}.csv"
        
        df.to_csv(filename, index=False)
        
        print(f"Exported: {filename}")
        print(f"Total gaps: {len(rows)}")
        print(f"Persons with gaps: {len(analysis_results)}")
        print()
        
        # Summary by type
        if len(rows) > 0:
            print("Gaps by type:")
            for gap_type, count in df['Gap_Type'].value_counts().items():
                print(f"  {gap_type}: {count}")
        
        try:
            from google.colab import files
            files.download(filename)
        except:
            pass

export_button.on_click(run_export)

display(HTML(f"""
<div style="{CONTAINER_STYLE}">
    <h3 style="color: {COLORS['text_primary']}; margin-top: 0;">📊 Export Summary</h3>
    <p>Export a CSV summary of all identified gaps for documentation.</p>
</div>
"""))
display(export_button)
display(export_output)